# Audit events to Splunk

End-to-end pipeline to build a consolidated audit Delta table `audit_events_splunk` from monitoring JSON, and export JSON Lines for Splunk.

In [ ]:
# Config and helpers
from pyspark.sql import functions as F

# Base folder for monitoring data under the default lakehouse Files
base_path = "Files/fabricmonitor"

# Optional OneLake diagnostics base
onelake_diag_base = "Files/DiagnosticLogs/OneLake/Workspaces"

# Ensure JSON writer omits null fields
spark.conf.set("spark.sql.jsonGenerator.ignoreNullFields", "true")

# Lets the MERGE add columns introduced by newer versions of this notebook to an existing table
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


def read_json_folder(relative_path: str):
    """Read multi-line JSON array files recursively from a subfolder of Files."""
    full_path = f"{relative_path}"
    return (
        spark.read
        .option("multiLine", "true")
        .option("recursiveFileLookup", "true")
        .json(full_path)
    )


def normalize_strings(df):
    """Trim strings and convert empty strings to null for all string columns."""
    from pyspark.sql.types import StringType
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.trim(F.col(c))))
    return df


def get_max_event_time():
    """Get max event_time from existing audit_events_splunk table, or None if it does not exist."""
    if spark.catalog._jcatalog.tableExists("audit_events_splunk"):
        max_ts = spark.read.table("audit_events_splunk").agg(F.max("event_time")).collect()[0][0]
        return max_ts
    return None


## Check that the module output exists

This notebook needs only the JSON files the monitor modules write to `Files/fabricmonitor/` (set `base_path` above to match `PATH_IN_LAKEHOUSE`). Run these modules first, in this order: **Activity, Graph, Catalog, FabricItems, ItemJobs, ControlPlane**. The next cell stops the run if `activity/` is empty and lists every other folder that is missing, with the module that writes it.

In [ ]:
# Same code as env/utility/audit_preflight.py
REQUIRED_SOURCES = {
    "activity": ("Activity", "the audit events themselves"),
}

OPTIONAL_SOURCES = {
    "graph": ("Graph", "user display names"),
    "lineage_edges": ("Catalog", "upstream/downstream lineage"),
    "catalog/scans": ("Catalog", "item names from the scanner"),
    "jobs": ("ItemJobs", "run history"),
    "itemdefinitions": ("ItemJobs", "sources and destinations"),
    "jobactivityruns": ("ItemJobs", "rows and bytes copied"),
    "controlplane": ("ControlPlane", "owner, item state, last run, failure reason and recent activity"),
}


def check_sources(has_data, required=None, optional=None):
    """has_data(folder) -> bool. Returns (missing_required, missing_optional, report_lines)."""
    required = REQUIRED_SOURCES if required is None else required
    optional = OPTIONAL_SOURCES if optional is None else optional
    lines = []
    missing_required = []
    missing_optional = []
    for group, sources, bucket in (("required", required, missing_required), ("optional", optional, missing_optional)):
        for folder, (module, effect) in sources.items():
            if has_data(folder):
                lines.append(f"OK       {folder}")
            else:
                bucket.append(folder)
                lines.append(f"MISSING  {folder} ({group}): run the {module} module; without it the table loses {effect}")
    return missing_required, missing_optional, lines


def folder_has_json(path):
    try:
        for entry in notebookutils.fs.ls(path):
            if entry.isDir:
                if folder_has_json(entry.path):
                    return True
            elif entry.name.endswith(".json") and entry.size > 2:
                return True
    except Exception:
        return False
    return False


missing_required, missing_optional, report = check_sources(lambda folder: folder_has_json(f"{base_path}/{folder}"))
print("\n".join(report))
if missing_required:
    raise Exception(f"No data under {base_path}/: {missing_required}. Run the monitor modules first.")
if missing_optional:
    print("\nContinuing: the table is built, but the columns listed above will be empty.")


## Inspect source schemas

## Load and filter activity events (incremental)

In [ ]:
# # Inspect schemas and sample rows for each source folder
# folders = [
#     ("activity", f"{base_path}/activity"),
#     ("graph", f"{base_path}/graph"),
#     ("catalog_scans", f"{base_path}/catalog/scans"),
#     ("lineage", f"{base_path}/lineage"),
#     ("jobs", f"{base_path}/jobs"),
#     ("itemdefinitions", f"{base_path}/itemdefinitions"),
#     ("jobactivityruns", f"{base_path}/jobactivityruns"),
# ]

# for name, path in folders:
#     print("====", name, "====")
#     try:
#         df = (
#             spark.read
#             .option("multiLine", "true")
#             .option("recursiveFileLookup", "true")
#             .json(path)
#         )
#         df.printSchema()
#         display(df.limit(2))
#     except Exception as e:
#         print(f"Failed to read {name} from {path}: {e}")


In [ ]:
# Set REBUILD = True once to drop the table and reprocess every event.
# Needed after changing enrichment logic, because incremental runs only enrich new events.
REBUILD = False
if REBUILD:
    spark.sql("DROP TABLE IF EXISTS audit_events_splunk")
    print("dropped audit_events_splunk - this run does a full rebuild")


In [ ]:
# Load activity events
activity_raw = read_json_folder(f"{base_path}/activity")
activity_raw = normalize_strings(activity_raw)

# Convert CreationTime to timestamp and compute incremental filter
activity = activity_raw.withColumn(
    "event_time",
    F.to_timestamp("CreationTime")
)

max_event_time = get_max_event_time()
if max_event_time is not None:
    activity = activity.filter(F.col("event_time") > F.lit(max_event_time))

# Prefer GUID-bearing item fields; ObjectId can be a display name for some operations.
zero_guid = "00000000-0000-0000-0000-000000000000"
activity = activity.withColumn(
    "item_id",
    F.lower(
        F.coalesce(
            *[
                F.when(F.lower(F.col(c)) != zero_guid, F.col(c))
                for c in ("ItemId", "ArtifactId", "DatasetId", "ObjectId")
            ]
        )
    )
)

# Standardize IDs to lower-case strings where present
id_cols = [
    "Id", "UserId", "UserKey", "WorkspaceId", "ObjectId", "ItemId", "DatasetId",
    "ArtifactId", "CapacityId", "RequestId", "ActivityId", "OrganizationId"
]
for c in id_cols:
    if c in activity.columns:
        activity = activity.withColumn(c, F.lower(F.col(c)))

# Event date partition column
activity = activity.withColumn("event_date", F.to_date("event_time"))

print(f"Activity events to process: {activity.count()}")


## Load dimension-like sources (graph, lineage, jobs, itemdefinitions, jobactivityruns)

In [ ]:
# 1) Graph users
try:
    graph_raw = read_json_folder(f"{base_path}/graph")
    graph_raw = normalize_strings(graph_raw)
    # Expecting memberId, displayName, userPrincipalName
    graph = (
        graph_raw
        .select(
            F.col("memberId"),
            F.col("displayName"),
            F.col("userPrincipalName")
        )
        .withColumn("user_id_lower", F.lower(F.col("userPrincipalName")))
        .withColumn("member_id_lower", F.lower(F.col("memberId")))
        .withColumn("display_name", F.col("displayName"))
        .dropDuplicates(["user_id_lower", "member_id_lower"])
    )
    print(f"Graph users: {graph.count()}")
except Exception as e:
    graph = None
    print(f"Graph folder missing or failed to load: {e}")

from pyspark.sql.window import Window

# 2) Lineage from lineage_edges/ folder (single authoritative source)
lineage_agg = None
try:
    edges_raw = (
        spark.read
        .option("multiLine", "true")
        .option("recursiveFileLookup", "true")
        .json(f"{base_path}/lineage_edges")
    )
    edges_raw = edges_raw.withColumn("file_name", F.input_file_name())
    if "edges" in edges_raw.columns:
        snapshots = (
            edges_raw.filter((F.col("schemaVersion") == 1) & (F.col("complete") == True))
            .select("file_name", F.explode("edges").alias("edge"))
            .select("file_name", "edge.*")
        )
        if "QueryItemId" in edges_raw.columns:
            legacy = edges_raw.filter(F.col("QueryItemId").isNotNull()).drop("edges")
            snapshots = snapshots.unionByName(legacy, allowMissingColumns=True)
        edges_raw = snapshots
    edges_raw = normalize_strings(edges_raw)

    # Normalize IDs and deduplicate (keep newest file)
    edges = (
        edges_raw
        .withColumn("query_item_id", F.lower(F.col("QueryItemId")))
        .withColumn("related_item_id", F.lower(F.col("RelatedItemId")))
    )

    w = (
        Window
        .partitionBy("query_item_id", "Direction", "related_item_id", "RelationType")
        .orderBy(F.col("file_name").desc())
    )

    edges_dedup = (
        edges
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # Label for related item: "Name (Type)" or "Id (Type)" if name is null
    related_label = F.concat(
        F.coalesce(F.col("RelatedItemName"), F.col("related_item_id")),
        F.lit(" ("),
        F.col("RelatedItemType"),
        F.lit(")"),
    )

    # Upstream aggregates
    upstream = (
        edges_dedup
        .filter(F.col("Direction") == F.lit("upstream"))
        .groupBy("query_item_id")
        .agg(
            F.array_sort(
                F.array_distinct(F.collect_list(related_label))
            ).alias("upstream_items"),
            F.size(
                F.array_distinct(F.collect_list(related_label))
            ).alias("upstream_count"),
            F.array_sort(
                F.array_distinct(F.collect_list("RelatedItemType"))
            ).alias("upstream_types"),
            F.array_sort(
                F.array_distinct(
                    F.collect_list(
                        F.when(
                            F.col("RelatedItemType") == F.lit("DataSource"),
                            related_label,
                        )
                    )
                )
            ).alias("data_sources"),
        )
    )

    # Downstream aggregates
    downstream = (
        edges_dedup
        .filter(F.col("Direction") == F.lit("downstream"))
        .groupBy("query_item_id")
        .agg(
            F.array_sort(
                F.array_distinct(F.collect_list(related_label))
            ).alias("downstream_items"),
            F.size(
                F.array_distinct(F.collect_list(related_label))
            ).alias("downstream_count"),
            F.array_sort(
                F.array_distinct(F.collect_list("RelatedItemType"))
            ).alias("downstream_types"),
        )
    )

    # Base lineage aggregates, arrays non-null
    lineage_agg = (
        upstream.alias("u")
        .join(downstream.alias("d"), "query_item_id", "outer")
        .select(
            F.col("query_item_id").alias("item_id"),
            F.coalesce("u.upstream_items", F.array().cast("array<string>")).alias("upstream_items"),
            F.coalesce("u.upstream_count", F.lit(0)).alias("upstream_count"),
            F.coalesce("u.upstream_types", F.array().cast("array<string>")).alias("upstream_types"),
            F.coalesce("d.downstream_items", F.array().cast("array<string>")).alias("downstream_items"),
            F.coalesce("d.downstream_count", F.lit(0)).alias("downstream_count"),
            F.coalesce("d.downstream_types", F.array().cast("array<string>")).alias("downstream_types"),
            F.coalesce("u.data_sources", F.array().cast("array<string>")).alias("data_sources"),
            F.lit(1).alias("has_lineage"),
        )
    )

    # PIPELINE ROLL-UP (up to 3 levels)
    # Parent->child relationships where parent has an upstream DataPipeline edge
    parent_child = (
        edges_dedup
        .filter(
            (F.col("Direction") == F.lit("upstream"))
            & (F.col("RelatedItemType") == F.lit("DataPipeline"))
        )
        .select(
            F.col("query_item_id").alias("parent_id"),
            F.col("related_item_id").alias("child_id"),
        )
        .dropDuplicates()
    )

    level1 = parent_child

    level2 = (
        level1.alias("l1")
        .join(
            parent_child.alias("pc"),
            F.col("l1.child_id") == F.col("pc.parent_id"),
            "inner",
        )
        .select(F.col("l1.parent_id"), F.col("pc.child_id"))
        .dropDuplicates()
    )

    level3 = (
        level2.alias("l2")
        .join(
            parent_child.alias("pc"),
            F.col("l2.child_id") == F.col("pc.parent_id"),
            "inner",
        )
        .select(F.col("l2.parent_id"), F.col("pc.child_id"))
        .dropDuplicates()
    )

    all_rel = (
        level1.union(level2).union(level3)
        .dropDuplicates()
    )

    # Attach child upstream_items and data_sources
    child_lineage = (
        lineage_agg
        .select(
            F.col("item_id").alias("child_id"),
            F.col("upstream_items"),
            F.col("data_sources"),
        )
    )

    parent_child_lineage = (
        all_rel.alias("rel")
        .join(child_lineage.alias("cl"), "child_id", "left")
    )

    rollup = (
        parent_child_lineage
        .groupBy("parent_id")
        .agg(
            F.array_sort(
                F.array_distinct(
                    F.flatten(F.collect_list("upstream_items"))
                )
            ).alias("rollup_upstream_items"),
            F.array_sort(
                F.array_distinct(
                    F.flatten(F.collect_list("data_sources"))
                )
            ).alias("rollup_data_sources"),
        )
    )

    if rollup is not None:
        lineage_agg = (
            lineage_agg.alias("base")
            .join(rollup.alias("r"), F.col("base.item_id") == F.col("r.parent_id"), "left")
            .select(
                F.col("base.item_id"),
                F.array_sort(
                    F.array_distinct(
                        F.concat(
                            F.col("base.upstream_items"),
                            F.coalesce(
                                F.col("r.rollup_upstream_items"),
                                F.array().cast("array<string>"),
                            ),
                        )
                    )
                ).alias("upstream_items"),
                F.size(
                    F.array_distinct(
                        F.concat(
                            F.col("base.upstream_items"),
                            F.coalesce(
                                F.col("r.rollup_upstream_items"),
                                F.array().cast("array<string>"),
                            ),
                        )
                    )
                ).alias("upstream_count"),
                F.col("base.upstream_types"),
                F.col("base.downstream_items"),
                F.col("base.downstream_count"),
                F.col("base.downstream_types"),
                F.array_sort(
                    F.array_distinct(
                        F.concat(
                            F.col("base.data_sources"),
                            F.coalesce(
                                F.col("r.rollup_data_sources"),
                                F.array().cast("array<string>"),
                            ),
                        )
                    )
                ).alias("data_sources"),
                F.col("base.has_lineage"),
            )
        )

    print(f"Lineage distinct items (lineage_edges): {lineage_agg.count()}")

except Exception as e:
    lineage_agg = None
    print(f"lineage_edges folder missing or failed to load: {e}")

# 3) Jobs
try:
    jobs_raw = read_json_folder(f"{base_path}/jobs")
    jobs_raw = normalize_strings(jobs_raw)
    jobs = (
        jobs_raw
        .withColumn("item_id", F.lower(F.col("ItemId")))
        .withColumn("start_time", F.to_timestamp("StartTimeUtc"))
        .withColumn("end_time", F.to_timestamp("EndTimeUtc"))
    )

    jobs_agg = (
        jobs
        .groupBy("item_id")
        .agg(
            F.countDistinct("JobInstanceId").alias("run_count"),
            F.sum(F.when(F.col("Status") == "Failed", 1).otherwise(0)).alias("failed_runs"),
            F.max("end_time").alias("last_run_time"),
            F.max(F.struct("end_time", "Status")).alias("last_run_struct"),
        )
        .withColumn("last_run_status", F.col("last_run_struct.Status"))
        .drop("last_run_struct")
    )
    print(f"Jobs distinct items: {jobs_agg.count()}")
except Exception as e:
    jobs_agg = None
    print(f"Jobs folder missing or failed to load: {e}")

# 4) Item definitions (sources/destinations) and names from definitions
itemdef_agg = None
itemdef_name_lookup = None
pipeline_rollup = None
try:
    itemdef_raw = read_json_folder(f"{base_path}/itemdefinitions")
    itemdef_raw = normalize_strings(itemdef_raw)
    itemdef = itemdef_raw.withColumn("item_id", F.lower(F.col("ItemId")))

    # Treat REST sources with no item name as "RestSource"
    source_name_col = F.when(
        F.col("TargetItemName").isNotNull(), F.col("TargetItemName")
    ).when(
        (F.col("TargetItemName").isNull()) & (F.col("EndpointType") == F.lit("Rest")),
        F.lit("RestSource")
    )

    sources = (
        itemdef
        .filter(F.col("Role") == F.lit("Source"))
        .groupBy("item_id")
        .agg(F.concat_ws("; ", F.collect_set(source_name_col)).alias("sources"))
    )

    destinations = (
        itemdef
        .filter(F.col("Role") == F.lit("Destination"))
        .groupBy("item_id")
        .agg(F.concat_ws("; ", F.collect_set("TargetItemName")).alias("destinations"))
    )

    itemdef_agg = sources.join(destinations, "item_id", "outer")
    print(f"Item definitions distinct items: {itemdef_agg.count()}")

    # Name lookup from itemdefinitions
    itemdef_name_lookup = (
        itemdef
        .select(F.col("item_id"), F.col("ItemName").alias("item_name_itemdef"))
        .where(F.col("ItemName").isNotNull())
        .dropDuplicates(["item_id"])
    )

    # Pipeline roll-up for sources/destinations remains as before
    try:
        # Start from jobactivityruns with ActivityType = 'ExecutePipeline'
        jr_raw_for_rollup = read_json_folder(f"{base_path}/jobactivityruns")
        jr_raw_for_rollup = normalize_strings(jr_raw_for_rollup)
        jr_exec = (
            jr_raw_for_rollup
            .filter(F.col("ActivityType") == F.lit("ExecutePipeline"))
            .select(
                F.lower(F.col("ItemId")).alias("parent_item_id"),
                F.col("ActivityName").alias("activity_name")
            )
            .dropDuplicates()
        )

        # Activity definitions in itemdefinitions (Role = 'Activity') tell us the child pipeline name
        activity_defs = (
            itemdef_raw
            .filter(F.col("Role") == F.lit("Activity"))
            .select(
                F.lower(F.col("ItemId")).alias("parent_item_id_def"),
                F.col("ActivityName").alias("activity_name_def"),
                F.col("TargetItemName").alias("child_pipeline_name")
            )
        )

        parent_child_names = (
            jr_exec.alias("jr")
            .join(
                activity_defs.alias("ad"),
                (F.col("jr.parent_item_id") == F.col("ad.parent_item_id_def"))
                & (F.col("jr.activity_name") == F.col("ad.activity_name_def")),
                "inner",
            )
            .select(
                F.col("jr.parent_item_id"),
                F.col("ad.child_pipeline_name")
            )
            .where(F.col("child_pipeline_name").isNotNull())
            .dropDuplicates()
        )

        # Resolve child pipeline names to their item_ids (ItemType = 'DataPipeline')
        child_pipelines = (
            itemdef_raw
            .filter(F.col("ItemType") == F.lit("DataPipeline"))
            .select(
                F.lower(F.col("ItemId")).alias("child_item_id"),
                F.col("ItemName").alias("child_item_name")
            )
        )

        parent_child_ids = (
            parent_child_names.alias("pc")
            .join(
                child_pipelines.alias("cp"),
                F.col("pc.child_pipeline_name") == F.col("cp.child_item_name"),
                "left",
            )
            .select(
                F.col("pc.parent_item_id"),
                F.col("cp.child_item_id"),
                F.col("cp.child_item_name"),
            )
            .dropDuplicates()
        )

        # Bring in child sources/destinations
        child_sd = (
            itemdef_agg
            .select(
                F.col("item_id").alias("child_item_id"),
                F.col("sources").alias("child_sources"),
                F.col("destinations").alias("child_destinations"),
            )
        )

        parent_child_enriched = (
            parent_child_ids.alias("m")
            .join(child_sd.alias("sd"), "child_item_id", "left")
        )

        pipeline_rollup = (
            parent_child_enriched
            .groupBy("parent_item_id")
            .agg(
                F.concat_ws(
                    "; ",
                    F.collect_set("child_sources"),
                ).alias("rollup_sources"),
                F.concat_ws(
                    "; ",
                    F.collect_set("child_destinations"),
                ).alias("rollup_destinations"),
                F.concat_ws(
                    "; ",
                    F.collect_set(
                        F.concat(F.col("child_item_name"), F.lit(" (DataPipeline)"))
                    ),
                ).alias("child_pipelines_downstream"),
            )
        )

        # Merge roll-up sources/destinations into base itemdef_agg
        if pipeline_rollup is not None:
            itemdef_agg = (
                itemdef_agg.alias("base")
                .join(
                    pipeline_rollup.alias("p"),
                    F.col("base.item_id") == F.col("p.parent_item_id"),
                    "left",
                )
                .select(
                    F.col("base.item_id"),
                    F.concat_ws(
                        "; ",
                        F.array_remove(
                            F.array(F.col("base.sources"), F.col("p.rollup_sources")),
                            F.lit(None),
                        ),
                    ).alias("sources"),
                    F.concat_ws(
                        "; ",
                        F.array_remove(
                            F.array(F.col("base.destinations"), F.col("p.rollup_destinations")),
                            F.lit(None),
                        ),
                    ).alias("destinations"),
                )
            )
    except Exception as pe:
        print(f"Pipeline roll-up skipped due to error: {pe}")

except Exception as e:
    print(f"Itemdefinitions folder missing or failed to load: {e}")

# 5) Job activity runs (volume)
jr_agg = None
try:
    jr_raw = read_json_folder(f"{base_path}/jobactivityruns")
    jr_raw = normalize_strings(jr_raw)
    jr = jr_raw.withColumn("item_id", F.lower(F.col("ItemId")))

    jr_agg = (
        jr
        .groupBy("item_id")
        .agg(
            F.sum(F.col("RowsCopied").cast("long")).alias("rows_copied"),
            F.sum(F.col("DataWrittenBytes").cast("long")).alias("bytes_written")
        )
    )
    print(f"Job activity runs distinct items: {jr_agg.count()}")
except Exception as e:
    print(f"Jobactivityruns folder missing or failed to load: {e}")

# 6) Name lookup from catalog scans and jobs
catalog_name_lookup = None
jobs_name_lookup = None

# (Catalog scans lineage is no longer used for lineage; keep any name lookup optional if needed)
try:
    scans_raw = read_json_folder(f"{base_path}/catalog/scans")
    scans_raw = normalize_strings(scans_raw)
    if "items" in scans_raw.columns:
        catalog_name_lookup = (
            scans_raw
            .select(
                F.lower(F.col("id")).alias("item_id"),
                F.col("name").alias("item_name_catalog"),
            )
            .where(F.col("item_name_catalog").isNotNull())
            .dropDuplicates(["item_id"])
        )
except Exception:
    catalog_name_lookup = None

if 'jobs' in globals():
    jobs_name_lookup = (
        jobs
        .select("item_id", F.col("ItemName").alias("item_name_jobs"))
        .where(F.col("ItemName").isNotNull())
        .dropDuplicates(["item_id"])
    )


In [ ]:
# Sources and destinations per item, rolled up from child pipelines to parents.
# Overrides itemdef_agg from the previous cell. Needs all_rel (parent/child pairs) from the lineage step.
itemdef = (normalize_strings(read_json_folder(f"{base_path}/itemdefinitions"))
           .withColumn("item_id", F.lower(F.col("ItemId"))))

# "RawLakehouse.dbo.dimproject (LakehouseTable)" or "RestSource [Get Items]"
obj = F.concat_ws(".", F.col("TargetItemName"), F.col("Schema"), F.col("Table"))
lbl = (F.when(F.length(obj) > 0, F.concat(obj, F.lit(" ("), F.col("EndpointType"), F.lit(")")))
        .otherwise(F.concat(F.col("EndpointType"), F.lit(" ["), F.coalesce(F.col("ActivityName"), F.lit("")), F.lit("]"))))

own_src = (itemdef.filter(F.col("Role") == "Source").groupBy("item_id")
           .agg(F.collect_set(lbl).alias("own_src")))
own_dst = (itemdef.filter(F.col("Role") == "Destination").groupBy("item_id")
           .agg(F.collect_set(lbl).alias("own_dst")))
own = own_src.join(own_dst, "item_id", "outer")

empty = F.array().cast("array<string>")

if 'all_rel' in globals():
    rel = all_rel.select(F.col("parent_id"), F.col("child_id"))
    roll = (rel.join(own.withColumnRenamed("item_id", "child_id"), "child_id", "inner")
            .groupBy("parent_id")
            .agg(F.array_distinct(F.flatten(F.collect_list(F.coalesce("own_src", empty)))).alias("roll_src"),
                 F.array_distinct(F.flatten(F.collect_list(F.coalesce("own_dst", empty)))).alias("roll_dst"))
            .withColumnRenamed("parent_id", "item_id"))
    allx = own.join(roll, "item_id", "outer")
else:
    allx = (own.withColumn("roll_src", F.lit(None).cast("array<string>"))
               .withColumn("roll_dst", F.lit(None).cast("array<string>")))


def merged(a, b):
    s = F.concat_ws("; ", F.array_sort(F.array_distinct(F.concat(F.coalesce(F.col(a), empty), F.coalesce(F.col(b), empty)))))
    return F.when(F.length(s) > 0, s)  # NULL instead of an empty string


itemdef_agg = allx.select("item_id",
                          merged("own_src", "roll_src").alias("sources"),
                          merged("own_dst", "roll_dst").alias("destinations"))
print("items with source/destination:",
      itemdef_agg.filter("sources IS NOT NULL OR destinations IS NOT NULL").count())


## Load the ControlPlane snapshot (owner, item state, last run, recent activity)

Reads the newest `controlplane/YYYY/MM/DD/` snapshot written by the ControlPlane module and reduces it to one row per item (`cp_agg`). The enrichment cell joins it on `item_id` with a left join, so a missing snapshot only leaves the `cp_*` columns empty.

In [ ]:
# Item-level fields from the newest ControlPlane snapshot
CP_COLUMNS = {
    "OwnerPrincipal": "cp_owner_principal",
    "State": "cp_item_state",
    "LastRunStatus": "cp_last_run_status",
    "LastRunEndUtc": "cp_last_run_end",
    "LastRunFailureReason": "cp_last_run_failure_reason",
    "RunsCollected": "cp_runs_collected",
    "FailedRunsCollected": "cp_failed_runs",
    "LastActivity": "cp_last_activity",
    "LastActivityTime": "cp_last_activity_time",
    "LastActivityUser": "cp_last_activity_user",
    "ActivityCount": "cp_activity_count_7d",
    "ActivityUsers": "cp_activity_users",
}

cp_agg = None
try:
    cp_raw = read_json_folder(f"{base_path}/controlplane").withColumn("_src", F.input_file_name())
    cp_raw = cp_raw.withColumn("_day", F.regexp_extract("_src", r"controlplane/(\d{4}/\d{2}/\d{2})/", 1))
    latest_day = cp_raw.agg(F.max("_day")).collect()[0][0]
    cp_raw = normalize_strings(cp_raw.filter(F.col("_day") == latest_day))

    present = [c for c in CP_COLUMNS if c in cp_raw.columns]
    cp_agg = (
        cp_raw.where(F.col("ItemId").isNotNull())
        .withColumn("item_id", F.lower(F.col("ItemId")))
        .groupBy("item_id")
        .agg(*[F.first(F.col(c).cast("string"), ignorenulls=True).alias(CP_COLUMNS[c]) for c in present])
    )
    for c in CP_COLUMNS.values():
        if c not in cp_agg.columns:
            cp_agg = cp_agg.withColumn(c, F.lit(None).cast("string"))
    print(f"ControlPlane snapshot {latest_day}: {cp_agg.count()} items")
except Exception as e:
    print(f"controlplane folder missing or failed to load, cp_* columns will be empty: {e}")


## Optional: Load OneLake diagnostic events

In [ ]:
# Optional OneLake diagnostics
from pyspark.sql.utils import AnalysisException

try:
    onelake_diag = (
        spark.read
        .option("multiLine", "true")
        .option("recursiveFileLookup", "true")
        .json(f"{onelake_diag_base}/")
    )
    onelake_diag = normalize_strings(onelake_diag)

    # Standardize IDs to lower-case
    onelake_diag = (
        onelake_diag
        .withColumn("user_id", F.lower(F.col("executingUPN")))
        .withColumn("item_id", F.lower(F.col("itemId")))
        .withColumn("workspace_id", F.lower(F.col("workspaceId")))
        .withColumn("event_time", F.to_timestamp("accessStartTime"))
        .withColumn("event_date", F.to_date("event_time"))
    )

    print(f"OneLake diagnostic events: {onelake_diag.count()}")
except Exception as e:
    onelake_diag = None
    print(f"OneLake diagnostics not available or failed to load: {e}")


## Build consolidated audit DataFrame

In [ ]:
# Base activity projection
activity_sel = activity.select(
    F.col("Id").alias("event_id"),
    F.col("event_time"),
    F.lit("Activity").alias("source"),
    F.col("Operation").alias("operation"),
    F.col("Activity").alias("activity"),
    F.col("ResultStatus").alias("result_status"),
    F.col("IsSuccess").cast("string").alias("is_success"),
    F.coalesce(
        F.col("ResultStatus"),
        F.when(F.lower(F.col("IsSuccess").cast("string")) == "true", F.lit("Succeeded"))
        .when(F.lower(F.col("IsSuccess").cast("string")) == "false", F.lit("Failed")),
        F.col("IsSuccess").cast("string"),
    ).alias("status"),
    F.col("UserId").alias("user_id"),
    F.col("UserType").alias("user_type"),
    F.col("ClientIP").alias("client_ip"),
    F.col("UserAgent").alias("user_agent"),
    F.col("WorkspaceId").alias("workspace_id"),
    F.col("WorkSpaceName").alias("workspace_name"),
    F.col("CapacityId").alias("capacity_id"),
    F.col("CapacityName").alias("capacity_name"),
    F.col("item_id"),
    F.col("ObjectDisplayName").alias("object_display_name"),
    F.coalesce(F.col("ObjectType"), F.col("ArtifactKind")).alias("object_type"),
    F.coalesce(
        F.col("ObjectDisplayName"),
        F.col("DatasetName"),
        F.col("ItemName"),
        F.col("ArtifactName")
    ).alias("item_name"),
    F.coalesce(F.col("ObjectType"), F.col("ArtifactKind")).alias("item_type"),
    # raw IDs for lineage join key
    F.col("ObjectId").alias("object_id"),
    F.col("ArtifactId").alias("artifact_id"),
    F.col("DatasetId").alias("dataset_id"),
    F.col("event_date")
)

activity_before_joins_count = activity_sel.count()
print(f"Activity rows before enrichments: {activity_before_joins_count}")

# Use the canonical key, with placeholder zero GUIDs already removed.
activity_sel = activity_sel.withColumn("event_item_key", F.col("item_id"))

# Use the resolved key for every item-level join below
activity_sel = activity_sel.withColumn("item_id", F.col("event_item_key"))

# Join user display name from graph: one row per lookup key, so no fan-out
if 'graph' in globals() and graph is not None:
    graph_lookup = (
        graph.select(F.col("user_id_lower").alias("k"), F.col("display_name"))
        .union(graph.select(F.col("member_id_lower").alias("k"), F.col("display_name")))
        .where(F.col("k").isNotNull() & F.col("display_name").isNotNull())
        .groupBy("k")
        .agg(F.first("display_name").alias("graph_display_name"))
    )
    activity_sel = (
        activity_sel.alias("a")
        .join(graph_lookup.alias("g"), F.lower(F.col("a.user_id")) == F.col("g.k"), "left")
        .withColumn("user_display_name", F.coalesce(F.col("graph_display_name"), F.col("user_id")))
        .drop("k", "graph_display_name")
    )
else:
    activity_sel = activity_sel.withColumn("user_display_name", F.col("user_id"))

# Join lineage (arrays + counts + data_sources)
if 'lineage_agg' in globals() and lineage_agg is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(
            lineage_agg.alias("l"),
            F.col("a.event_item_key") == F.col("l.item_id"),
            "left",
        )
        .select(
            "a.*",
            "l.upstream_items",
            "l.upstream_count",
            "l.upstream_types",
            "l.downstream_items",
            "l.downstream_count",
            "l.downstream_types",
            "l.data_sources",
            "l.has_lineage",
        )
    )
else:
    activity_sel = (
        activity_sel
        .withColumn("upstream_items", F.array().cast("array<string>"))
        .withColumn("upstream_count", F.lit(0))
        .withColumn("upstream_types", F.array().cast("array<string>"))
        .withColumn("downstream_items", F.array().cast("array<string>"))
        .withColumn("downstream_count", F.lit(0))
        .withColumn("downstream_types", F.array().cast("array<string>"))
        .withColumn("data_sources", F.array().cast("array<string>"))
        .withColumn("has_lineage", F.lit(None).cast("int"))
    )

# Join jobs
if 'jobs_agg' in globals() and jobs_agg is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(jobs_agg.alias("j"), "item_id", "left")
    )
else:
    activity_sel = (
        activity_sel
        .withColumn("run_count", F.lit(None))
        .withColumn("failed_runs", F.lit(None))
        .withColumn("last_run_time", F.lit(None))
        .withColumn("last_run_status", F.lit(None))
    )

# Join item definitions (sources/destinations)
if 'itemdef_agg' in globals() and itemdef_agg is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(itemdef_agg.alias("d"), "item_id", "left")
    )
else:
    activity_sel = activity_sel.withColumn("sources", F.lit(None)).withColumn("destinations", F.lit(None))

# Join job activity runs (volume)
if 'jr_agg' in globals() and jr_agg is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(jr_agg.alias("v"), "item_id", "left")
    )
else:
    activity_sel = activity_sel.withColumn("rows_copied", F.lit(None)).withColumn("bytes_written", F.lit(None))

# Join name lookups from catalog, jobs, and itemdefinitions for better item_name
if 'catalog_name_lookup' in globals() and catalog_name_lookup is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(catalog_name_lookup.alias("c"), "item_id", "left")
    )
else:
    activity_sel = activity_sel.withColumn("item_name_catalog", F.lit(None))

if 'jobs_name_lookup' in globals() and jobs_name_lookup is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(jobs_name_lookup.alias("jname"), "item_id", "left")
    )
else:
    activity_sel = activity_sel.withColumn("item_name_jobs", F.lit(None))

if 'itemdef_name_lookup' in globals() and itemdef_name_lookup is not None:
    activity_sel = (
        activity_sel.alias("a")
        .join(itemdef_name_lookup.alias("dname"), "item_id", "left")
    )
else:
    activity_sel = activity_sel.withColumn("item_name_itemdef", F.lit(None))

activity_sel = activity_sel.withColumn(
    "item_name",
    F.coalesce(
        F.col("item_name"),
        F.col("item_name_catalog"),
        F.col("item_name_jobs"),
        F.col("item_name_itemdef")
    )
)

# Join the newest ControlPlane snapshot (owner, item state, last run, recent activity)
if 'cp_agg' in globals() and cp_agg is not None:
    activity_sel = activity_sel.alias("a").join(cp_agg.alias("cp"), "item_id", "left")
else:
    for c in CP_COLUMNS.values():
        activity_sel = activity_sel.withColumn(c, F.lit(None).cast("string"))

# lineage_note
activity_sel = activity_sel.withColumn(
    "lineage_note",
    F.when(
        F.col("event_item_key").isNull(),
        F.lit("no item id on event"),
    ).when(
        F.col("event_item_key").isNotNull() & F.col("has_lineage").isNull(),
        F.lit("item not in lineage_edges"),
    ).when(
        F.col("has_lineage").isNotNull()
        & (F.col("upstream_count") == 0)
        & (F.col("downstream_count") == 0),
        F.lit("no relations in scanner"),
    ).otherwise(F.lit("ok")),
)

activity_after_joins_count = activity_sel.count()
print(f"Activity rows after enrichments: {activity_after_joins_count}")

if activity_before_joins_count != activity_after_joins_count:
    raise Exception("Row count changed after enrichments; left joins violated.")

activity_audit = activity_sel


## Append OneLake diagnostics (if available)

In [ ]:
if 'onelake_diag' in globals() and onelake_diag is not None and onelake_diag.columns:
    onelake_sel = onelake_diag.select(
        F.sha2(F.concat_ws("|", F.col("operationName"), F.col("accessStartTime"), F.col("itemId")), 256).alias("event_id"),
        F.col("event_time"),
        F.lit("OneLake").alias("source"),
        F.col("operationName").alias("operation"),
        F.lit(None).cast("string").alias("activity"),
        F.col("httpStatusCode").cast("string").alias("result_status"),
        F.lit(None).cast("string").alias("is_success"),
        F.col("httpStatusCode").cast("string").alias("status"),
        F.col("user_id"),
        F.lit(None).cast("string").alias("user_type"),
        F.col("callerIPAddress").alias("client_ip"),
        F.lit(None).cast("string").alias("user_agent"),
        F.col("workspace_id"),
        F.lit(None).cast("string").alias("workspace_name"),
        F.lit(None).cast("string").alias("capacity_id"),
        F.lit(None).cast("string").alias("capacity_name"),
        F.col("item_id"),
        F.col("itemType").alias("item_type"),
        F.lit(None).cast("string").alias("item_name"),
        F.col("event_date"),
        F.lit(None).cast("string").alias("user_display_name"),
        F.lit(None).cast("string").alias("upstream_items"),
        F.lit(None).cast("string").alias("downstream_items"),
        F.lit(None).cast("string").alias("last_run_status"),
        F.lit(None).cast("timestamp").alias("last_run_time"),
        F.lit(None).cast("long").alias("run_count"),
        F.lit(None).cast("long").alias("failed_runs"),
        F.lit(None).cast("string").alias("sources"),
        F.lit(None).cast("string").alias("destinations"),
        F.lit(None).cast("long").alias("rows_copied"),
        F.lit(None).cast("long").alias("bytes_written")
    )

    audit_all = activity_audit.unionByName(onelake_sel, allowMissingColumns=True)
else:
    audit_all = activity_audit

print(f"Total audit events (including OneLake if any): {audit_all.count()}")


## Write Delta table with MERGE (incremental, partitioned by event_date)

In [ ]:
from delta.tables import DeltaTable

# Before writing, ensure lineage columns are stored as JSON-like strings in Delta
# Use cast("string") so this works whether the column is already a string or an array

audit_all_for_delta = (
    audit_all
    .withColumn("upstream_items", F.col("upstream_items").cast("string"))
    .withColumn("downstream_items", F.col("downstream_items").cast("string"))
)

# Write using MERGE on event_id
spark.sql("CREATE DATABASE IF NOT EXISTS default")

if not spark.catalog._jcatalog.tableExists("audit_events_splunk"):
    # First load: create table
    (
        audit_all_for_delta
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_date")
        .saveAsTable("audit_events_splunk")
    )
else:
    if "status" not in spark.read.table("audit_events_splunk").columns:
        spark.sql("ALTER TABLE audit_events_splunk ADD COLUMNS (status STRING)")
    target = DeltaTable.forName(spark, "audit_events_splunk")
    target.alias("t").merge(
        audit_all_for_delta.alias("s"),
        "t.event_id = s.event_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"audit_events_splunk row count: {spark.read.table('audit_events_splunk').count()}")


## Export JSON Lines for Splunk

In [ ]:
# Export per event_date as JSON Lines
export_base = "Files/splunk_export"

# Get distinct dates we just processed
processed_dates = audit_all.select("event_date").distinct().collect()

for row in processed_dates:
    d = row[0]
    if d is None:
        continue
    yyyy = d.strftime("%Y")
    mm = d.strftime("%m")
    dd = d.strftime("%d")

    out_path = f"{export_base}/{yyyy}/{mm}/{dd}/audit_events.jsonl"

    (
        audit_all
        .filter(F.col("event_date") == F.lit(d))
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("compression", "none")
        .json(out_path)
    )

print(f"Exported JSON Lines to {export_base}/YYYY/MM/DD/audit_events.jsonl")


## Data quality checks

In [ ]:
# Row counts per source
source_counts = audit_all.groupBy("source").count()
print("Row counts per source:")
for r in source_counts.collect():
    print(f"  {r['source']}: {r['count']}")

# Missing display name or item name
missing_user_display = audit_all.filter(F.col("user_display_name").isNull()).count()
missing_item_name = audit_all.filter(F.col("item_name").isNull()).count()

print(f"Events with missing user_display_name: {missing_user_display}")
print(f"Events with missing item_name: {missing_item_name}")

# Missing item_name grouped by operation
print("\nMissing item_name by operation:")
missing_by_op = (
    audit_all
    .filter(F.col("item_name").isNull())
    .groupBy("operation")
    .count()
)
for r in missing_by_op.collect():
    print(f"  {r['operation']}: {r['count']}")

# Non-null percentage for enrichment columns
enrichment_cols = [
    "user_display_name",
    "upstream_items",
    "downstream_items",
    "last_run_status",
    "last_run_time",
    "run_count",
    "failed_runs",
    "sources",
    "destinations",
    "rows_copied",
    "bytes_written",
    "lineage_note",
]

total = audit_all.count()
print(f"\nTotal events: {total}")

for col_name in enrichment_cols:
    non_null = audit_all.filter(F.col(col_name).isNotNull()).count()
    pct = (non_null / total * 100) if total > 0 else 0
    print(f"{col_name}: {non_null} non-null ({pct:.2f}%)")

# Lineage diagnostics
print("\nEvents by lineage_note:")
for r in (
    audit_all
    .groupBy("lineage_note")
    .count()
    .orderBy(F.desc("count"))
    .collect()
):
    print(f"  {r['lineage_note']}: {r['count']}")

print("\nSample events with lineage:")
sample = (
    audit_all
    .select("item_name", "upstream_items", "downstream_items", "lineage_note")
    .limit(10)
)
display(sample)

print("\nTop 20 item names with lineage_note = 'item not in lineage_edges':")
missing_items = (
    audit_all
    .filter(F.col("lineage_note") == "item not in lineage_edges")
    .groupBy("item_name")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)
display(missing_items)
